# 05 · Procurement Twin (Step 4)

The **Procurement twin** — the inbound supply network: which suppliers deliver which materials,
when, and how reliably.

**Owns:** open/past-due POs, expected receipts, supplier OTIF / in-full / on-time, lead-time
& variability, rejected quantity, invoice accuracy, supplier risk.

**Inputs:** `gold.vendorperformance` (rich pre-computed PO history).
**Outputs (monitoring):** `procurement_twin_current`, `procurement_twin_metrics_timeseries`.

Grain: supplier × material × plant. Idempotent overwrite.

## Config

In [ ]:
dbutils.widgets.text("source_catalog", "mickmickstableaws_catalog", "Source catalog")
dbutils.widgets.text("source_schema", "gold", "Source schema")
dbutils.widgets.text("target_catalog", "mickmickstableaws_catalog", "Target catalog")
dbutils.widgets.text("monitoring_schema", "monitoring", "Monitoring schema")
dbutils.widgets.text("window_days", "365", "Recent window for current-state rollups")
dbutils.widgets.text("risk_otif_threshold", "0.80", "OTIF below this flags supplier risk")
dbutils.widgets.text("risk_rejection_threshold", "0.05", "Rejection rate above this flags risk")

import uuid
from pyspark.sql import functions as F

SRC = f"{dbutils.widgets.get('source_catalog').strip()}.{dbutils.widgets.get('source_schema').strip()}"
FQ_MON = f"{dbutils.widgets.get('target_catalog').strip()}.{dbutils.widgets.get('monitoring_schema').strip()}"
WINDOW = int(dbutils.widgets.get("window_days"))
RISK_OTIF = float(dbutils.widgets.get("risk_otif_threshold"))
RISK_REJ = float(dbutils.widgets.get("risk_rejection_threshold"))
REFRESH_ID = str(uuid.uuid4())

VP = f"{SRC}.vendorperformance"
AS_OF_SQL = f"(SELECT MAX(PurchasingDocumentDate_BEDAT) FROM {VP})"
CUR_OUT = f"{FQ_MON}.procurement_twin_current"
TS_OUT  = f"{FQ_MON}.procurement_twin_metrics_timeseries"
print(f"Source: {VP}\nOutputs: {CUR_OUT} , {TS_OUT}\nWindow: {WINDOW}d")

## 1. Build `procurement_twin_current`
Recent-window supplier reliability + open order book per supplier × material × plant.

In [ ]:
current = spark.sql(f"""
SELECT *,
  (COALESCE(otif_rate, 0) < {RISK_OTIF} OR COALESCE(rejection_rate, 0) > {RISK_REJ}) AS supplier_risk_flag
FROM (
  SELECT
    Client_MANDT AS mandt, VendorAccountNumber_LIFNR AS supplier,
    MaterialNumber_MATNR AS material, Plant_WERKS AS plant,
    MAX(NAME1) AS supplier_name, MAX(CountryKey_LAND1) AS supplier_country,
    MAX(MaterialText_MAKTX) AS material_text, MAX(MaterialGroup_MATKL) AS material_group, MAX(UoM_MEINS) AS uom,
    COUNT(*) AS po_line_count,
    SUM(POQuantity_MENGE) AS po_qty, SUM(GoodsReceiptQuantity) AS received_qty, SUM(InvoiceQuantity) AS invoiced_qty,
    SUM(GREATEST(POQuantity_MENGE - GoodsReceiptQuantity, 0)) AS open_po_qty,
    SUM(CASE WHEN GoodsReceiptQuantity < POQuantity_MENGE THEN POQuantity_MENGE - GoodsReceiptQuantity END) AS expected_open_receipt_qty,
    MIN(CASE WHEN GoodsReceiptQuantity < POQuantity_MENGE AND ItemDeliveryDate_EINDT >= {AS_OF_SQL}
             THEN ItemDeliveryDate_EINDT END) AS next_expected_receipt_date,
    AVG(CAST(VendorOnTimeInFullDelivery AS DOUBLE)) AS otif_rate,
    AVG(CAST(VendorOnTimeDelivery AS DOUBLE)) AS on_time_rate,
    AVG(CAST(VendorInFullDelivery AS DOUBLE)) AS in_full_rate,
    AVG(CAST(VendorInvoiceAccuracy AS DOUBLE)) AS invoice_accuracy_rate,
    AVG(VendorCycleTimeInDays) AS avg_lead_time_days,
    STDDEV_SAMP(VendorCycleTimeInDays) AS lead_time_std_days,
    SUM(RejectedQuantity) AS rejected_qty,
    CASE WHEN SUM(POQuantity_MENGE) > 0 THEN SUM(RejectedQuantity) / SUM(POQuantity_MENGE) END AS rejection_rate,
    AVG(CAST(PastDueOrOpenItems AS DOUBLE)) AS past_due_rate,
    CAST({AS_OF_SQL} AS DATE) AS as_of_date
  FROM {VP}
  WHERE MaterialNumber_MATNR IS NOT NULL AND VendorAccountNumber_LIFNR IS NOT NULL
    AND PurchasingDocumentDate_BEDAT > date_sub({AS_OF_SQL}, {WINDOW})
  GROUP BY 1,2,3,4
)
""")
current = (current.withColumn("refresh_id", F.lit(REFRESH_ID)).withColumn("generated_at", F.current_timestamp()))
(current.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(CUR_OUT))
spark.sql(f"COMMENT ON TABLE {CUR_OUT} IS 'Procurement twin current state per supplier x material x plant: reliability (OTIF/lead time/rejection/invoice accuracy), open POs, expected receipts, risk.'")
print(f"Wrote {current.count():,} supplier-material-plant rows to {CUR_OUT}")

## 2. Build `procurement_twin_metrics_timeseries`
Supplier reliability & inbound volume over time (weekly by purchasing document date).

In [ ]:
timeseries = spark.sql(f"""
SELECT Client_MANDT AS mandt, VendorAccountNumber_LIFNR AS supplier,
       MaterialNumber_MATNR AS material, Plant_WERKS AS plant,
       date_trunc('week', PurchasingDocumentDate_BEDAT) AS period_start,
  COUNT(*) AS po_line_count,
  SUM(POQuantity_MENGE) AS po_qty, SUM(GoodsReceiptQuantity) AS received_qty, SUM(RejectedQuantity) AS rejected_qty,
  AVG(CAST(VendorOnTimeInFullDelivery AS DOUBLE)) AS otif_rate,
  AVG(CAST(VendorOnTimeDelivery AS DOUBLE)) AS on_time_rate,
  AVG(VendorCycleTimeInDays) AS avg_lead_time_days
FROM {VP}
WHERE PurchasingDocumentDate_BEDAT IS NOT NULL AND MaterialNumber_MATNR IS NOT NULL
GROUP BY 1,2,3,4,5
""")
timeseries = (timeseries.withColumn("refresh_id", F.lit(REFRESH_ID)).withColumn("generated_at", F.current_timestamp()))
(timeseries.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TS_OUT))
spark.sql(f"COMMENT ON TABLE {TS_OUT} IS 'Procurement twin metric time-series per supplier x material x plant x week: PO/received/rejected qty, OTIF, on-time, lead time.'")
print(f"Wrote {timeseries.count():,} rows to {TS_OUT}")

## Column descriptions
Apply human-readable descriptions to every output column (persisted on rebuild).

In [ ]:
# Column descriptions — re-applied after the overwrite writes so they persist in the
# catalog on every rebuild. EXTEND for your business.
COLUMN_COMMENTS = {
    CUR_OUT: {
        "mandt": "SAP client identifier",
        "supplier": "Vendor / supplier account number",
        "material": "Material / SKU number",
        "plant": "Plant",
        "supplier_name": "Supplier name",
        "supplier_country": "Supplier country key",
        "material_text": "Material description",
        "material_group": "Material group / category",
        "uom": "Unit of measure",
        "po_line_count": "Number of purchase-order lines in the window",
        "po_qty": "Total purchase-order quantity",
        "received_qty": "Total goods-receipt quantity",
        "invoiced_qty": "Total invoiced quantity",
        "open_po_qty": "Open PO quantity not yet received",
        "expected_open_receipt_qty": "Outstanding quantity expected from open POs",
        "next_expected_receipt_date": "Earliest expected receipt date among open POs",
        "otif_rate": "On-time-in-full delivery rate",
        "on_time_rate": "On-time delivery rate",
        "in_full_rate": "In-full delivery rate",
        "invoice_accuracy_rate": "Invoice accuracy rate",
        "avg_lead_time_days": "Average supplier lead time in days",
        "lead_time_std_days": "Std deviation of supplier lead time in days",
        "rejected_qty": "Total rejected quantity",
        "rejection_rate": "Rejected quantity as a share of PO quantity",
        "past_due_rate": "Share of lines flagged past-due or open",
        "as_of_date": "As-of date of this twin snapshot",
        "supplier_risk_flag": "True when OTIF is below or rejection above the configured thresholds",
        "refresh_id": "Unique id of the refresh run",
        "generated_at": "Timestamp this row was generated",
    },
    TS_OUT: {
        "mandt": "SAP client identifier",
        "supplier": "Vendor / supplier account number",
        "material": "Material / SKU number",
        "plant": "Plant",
        "period_start": "Start of the week bucket (purchasing document date)",
        "po_line_count": "Purchase-order lines in the week",
        "po_qty": "Purchase-order quantity in the week",
        "received_qty": "Received quantity in the week",
        "rejected_qty": "Rejected quantity in the week",
        "otif_rate": "On-time-in-full rate in the week",
        "on_time_rate": "On-time rate in the week",
        "avg_lead_time_days": "Average lead time in days in the week",
        "refresh_id": "Unique id of the refresh run",
        "generated_at": "Timestamp this row was generated",
    },
}
for _tbl, _cols in COLUMN_COMMENTS.items():
    for _c, _desc in _cols.items():
        spark.sql(f"ALTER TABLE {_tbl} ALTER COLUMN {_c} COMMENT '{_desc}'")
print(f"Applied column descriptions to {CUR_OUT} and {TS_OUT}")

## 3. Verify

In [ ]:
print("=== procurement_twin_current (riskiest suppliers) ===")
display(spark.sql(f"""
SELECT supplier, supplier_name, material, plant, po_line_count, ROUND(otif_rate,2) otif,
       ROUND(avg_lead_time_days,1) lead_days, ROUND(rejection_rate,3) rej_rate, open_po_qty,
       next_expected_receipt_date, supplier_risk_flag
FROM {CUR_OUT} ORDER BY otif_rate ASC NULLS FIRST LIMIT 25
"""))
display(spark.sql(f"SELECT SUM(CAST(supplier_risk_flag AS INT)) at_risk, COUNT(*) rows, ROUND(AVG(otif_rate),3) avg_otif, ROUND(AVG(avg_lead_time_days),1) avg_lead FROM {CUR_OUT}"))
display(spark.sql(f"SELECT COUNT(*) rows, COUNT(DISTINCT supplier||material||plant) series, MIN(period_start) first_p, MAX(period_start) last_p FROM {TS_OUT}"))

## Next
Procurement twin live. Next: **Production** (`06_production_twin`) — WIP, yield/scrap,
work-centre load, bottleneck pressure, cost/schedule variance.